# Macro Dashboard EDA: `eda_observations.csv`

This notebook explores the cleaned EDA dataset generated from the macro dashboard database.

Main goals:
- Inspect data quality, missing values, types, and coverage.
- Study distributions for key macro themes: inflation, GDP/growth, and unemployment/labor.
- Visualize time-series trends, rolling statistics, and stationarity.
- Examine correlations, lead-lag relationships, and potential multicollinearity.
- Compare countries, central banks, categories, and frequencies.
- Run PCA to identify broad latent macro factors.

The notebook is designed to be run from the repository root.

## 1. Setup and Data Loading

We load the EDA table from CSV, enforce expected data types, and inspect basic data health.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import adfuller, kpss

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

DATA_PATH = Path("data/eda/eda_observations.csv")
assert DATA_PATH.exists(), f"Missing {DATA_PATH}. Run: docker exec -w /app macro_dashboard_app python -m scripts.build_eda_dataset"

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["release_timestamp_utc"] = pd.to_datetime(df["release_timestamp_utc"], errors="coerce", utc=True)
df["retrieved_at_utc"] = pd.to_datetime(df["retrieved_at_utc"], errors="coerce", utc=True)

numeric_cols = [
    "value", "estimate_value", "previous_value", "surprise_value",
    "value_zscore", "value_minmax", "value_normalized",
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df["is_outlier_iqr"] = df["is_outlier_iqr"].astype(bool)
df = df.dropna(subset=["date", "value", "central_bank_code", "indicator_key"])

print(f"Rows: {len(df):,}")
print(f"Series: {df[['central_bank_code', 'indicator_key']].drop_duplicates().shape[0]:,}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
df.head()

Rows: 15,100
Series: 293
Date range: 2019-10-01 to 2026-04-22


,source_release_id,date,country,central_bank_code,currency_code,country_name,central_bank,indicator,indicator_key,primary_category,importance,frequency,period_label,release_timestamp_utc,retrieved_at_utc,value,estimate_value,previous_value,surprise_value,value_zscore,value_minmax,value_normalized,is_outlier_iqr,year,quarter,month
0,26901,2019-10-01,UK,BOE,GBP,United Kingdom,Bank of England,Employment Change,employment_change,Labor,2,monthly,Oct,2020-01-21 09:30:00+00:00,2026-04-21 07:39:40.535256+00:00,208.0000,104.0000,24.0000,104.0000,1.1240,0.7218,1.1240,False,2019,4,10
1,18461,2019-10-01,JP,BOJ,JPY,Japan,Bank of Japan,GDP (QoQ),gdp_qoq,Growth,1,quarterly,Q4,2020-03-08 23:50:00+00:00,2026-04-21 07:39:17.335659+00:00,-1.8000,-1.7000,NaN,-0.1000,-0.8487,0.4621,-0.8487,False,2019,4,10
2,12436,2019-10-01,EU,ECB,EUR,Eurozone,European Central Bank,GDP (QoQ),gdp_qoq,Growth,1,quarterly,Q4,2020-03-10 10:00:00+00:00,2026-04-21 07:38:56.814612+00:00,0.1000,0.1000,0.3000,0.0000,-0.0291,0.4897,-0.0291,False,2019,4,10
3,33635,2019-10-01,US,FED,USD,United States,Federal Reserve,Core PCE Prices (QoQ),core_pce_prices_qoq,Inflation,2,quarterly,Q4,2020-03-26 12:30:00+00:00,2026-04-21 07:39:55.426303+00:00,1.3000,1.2000,2.1000,0.1000,-1.1954,0.3043,-1.1954,False,2019,4,10
4,33630,2019-10-01,US,FED,USD,United States,Federal Reserve,PCE Prices (QoQ),pce_prices_qoq,Inflation,2,quarterly,Q4,2020-03-26 12:30:00+00:00,2026-04-21 07:39:55.418914+00:00,1.4000,1.3000,1.5000,0.1000,-0.9704,0.3371,-0.9704,False,2019,4,10


In [2]:
missing_summary = (
    df.isna().sum()
    .rename("missing_count")
    .to_frame()
    .assign(missing_pct=lambda x: 100 * x["missing_count"] / len(df))
    .sort_values("missing_count", ascending=False)
)
missing_summary.head(20)

,missing_count,missing_pct
surprise_value,2155,14.2715
estimate_value,2155,14.2715
previous_value,238,1.5762
period_label,153,1.0132
currency_code,0,0.0000
country_name,0,0.0000
country,0,0.0000
central_bank_code,0,0.0000
source_release_id,0,0.0000
date,0,0.0000


### Interpretation

The EDA dataset is already a cleaned derivative of the raw database. Rows without usable dates or actual values were excluded before this notebook. Remaining missingness is expected mainly in optional fields such as estimates, previous values, and surprises.

## 2. Key Indicator Selection

We create focused views for inflation, GDP/growth, and unemployment/labor indicators. These are broad filters, so review the selected indicator lists before modeling.

In [3]:
def contains_any(series: pd.Series, tokens: list[str]) -> pd.Series:
    pattern = "|".join(tokens)
    return series.str.contains(pattern, case=False, regex=True, na=False)

inflation_mask = (
    df["primary_category"].str.contains("Inflation", case=False, na=False)
    | contains_any(df["indicator"], ["cpi", "inflation", "ppi", "price"])
)
gdp_mask = contains_any(df["indicator"], ["gdp", "gross domestic", "growth"])
unemployment_mask = contains_any(df["indicator"], ["unemployment", "jobless", "claimant"])

key_df = df.assign(
    key_theme=np.select(
        [inflation_mask, gdp_mask, unemployment_mask],
        ["Inflation", "GDP/Growth", "Unemployment"],
        default="Other",
    )
)
key_focus = key_df[key_df["key_theme"] != "Other"].copy()

indicator_inventory = (
    key_focus.groupby(["key_theme", "central_bank_code", "indicator_key", "indicator"])
    .agg(observations=("value", "count"), first_date=("date", "min"), last_date=("date", "max"))
    .reset_index()
    .sort_values(["key_theme", "central_bank_code", "indicator_key"])
)
indicator_inventory.head(40)

,key_theme,central_bank_code,indicator_key,indicator,observations,first_date,last_date
0,GDP/Growth,BOC,gdp_mom,GDP (MoM),71,2019-11-01,2026-02-01
1,GDP/Growth,BOE,gdp_mom,GDP (MoM),69,2019-11-01,2026-02-01
2,GDP/Growth,BOE,gdp_qoq,GDP (QoQ),24,2020-01-01,2025-10-01
3,GDP/Growth,BOE,gdp_yoy,GDP (YoY),73,2019-11-01,2026-02-01
4,GDP/Growth,BOE,niesr_monthly_gdp_tracker,NIESR Monthly GDP Tracker,64,2020-02-01,2026-03-01
5,GDP/Growth,BOJ,gdp_qoq,GDP (QoQ),25,2019-10-01,2025-10-01
6,GDP/Growth,ECB,gdp_qoq,GDP (QoQ),24,2019-10-01,2025-10-01
7,GDP/Growth,SNB,gdp_qoq,GDP (QoQ),23,2019-10-01,2025-10-01
8,Inflation,BOC,core_cpi_mom,Core CPI (MoM),45,2022-03-01,2026-03-01
9,Inflation,BOC,core_cpi_yoy,Core CPI (YoY),76,2019-12-01,2026-03-01


## 3. Univariate Analysis

We inspect summary statistics, normalized histograms, density-style histograms, and box plots. Normalized values are z-scores by central bank and indicator, so `0` is normal for that series, `+2` is unusually high, and `-2` is unusually low.

In [4]:
summary_stats = (
    key_focus.groupby(["key_theme", "central_bank_code", "indicator_key", "indicator"])
    .agg(
        observations=("value", "count"),
        mean=("value", "mean"),
        median=("value", "median"),
        std=("value", "std"),
        min=("value", "min"),
        max=("value", "max"),
        skew=("value", "skew"),
        outliers=("is_outlier_iqr", "sum"),
    )
    .reset_index()
    .sort_values(["key_theme", "central_bank_code", "indicator_key"])
)
summary_stats.head(30)

,key_theme,central_bank_code,indicator_key,indicator,observations,mean,median,std,min,max,skew,outliers
0,GDP/Growth,BOC,gdp_mom,GDP (MoM),71,0.1366,0.2000,1.9533,-11.6000,6.5000,-3.2415,5
1,GDP/Growth,BOE,gdp_mom,GDP (MoM),69,0.0261,0.2000,3.0015,-20.4000,8.7000,-4.3606,6
2,GDP/Growth,BOE,gdp_qoq,GDP (QoQ),24,0.2042,0.1500,5.4835,-19.8000,16.0000,-1.1310,4
3,GDP/Growth,BOE,gdp_yoy,GDP (YoY),73,0.5685,1.0000,7.6899,-24.5000,27.6000,-0.0969,23
4,GDP/Growth,BOE,niesr_monthly_gdp_tracker,NIESR Monthly GDP Tracker,64,0.1906,0.2000,4.4300,-21.2000,15.2000,-2.0191,8
5,GDP/Growth,BOJ,gdp_qoq,GDP (QoQ),25,0.0200,0.1000,2.1446,-7.9000,5.3000,-1.5260,2
6,GDP/Growth,ECB,gdp_qoq,GDP (QoQ),24,0.2083,0.2500,3.7287,-11.8000,12.5000,0.0782,5
7,GDP/Growth,SNB,gdp_qoq,GDP (QoQ),23,0.1870,0.3000,2.4664,-8.2000,7.2000,-0.8608,5
8,Inflation,BOC,core_cpi_mom,Core CPI (MoM),45,0.2578,0.3000,0.3292,-0.5000,1.0000,-0.2435,0
9,Inflation,BOC,core_cpi_yoy,Core CPI (YoY),76,2.9039,2.6000,1.5044,0.7000,6.2000,0.7802,0


In [5]:
fig = px.histogram(
    key_focus,
    x="value_normalized",
    color="key_theme",
    facet_col="central_bank_code",
    facet_col_wrap=2,
    nbins=60,
    histnorm="probability density",
    title="Normalized Distributions for Inflation, GDP/Growth, and Unemployment Indicators",
)
fig.update_layout(height=1100)
fig.show()

In [6]:
fig = px.box(
    key_focus,
    x="central_bank_code",
    y="value_normalized",
    color="key_theme",
    points="outliers",
    title="Outlier Structure by Central Bank and Key Macro Theme",
)
fig.show()

### Interpretation checklist

- Wide distributions indicate volatile indicators or regimes with sharp macro moves.
- Long right tails in inflation mean unusually high inflation prints.
- Long tails in unemployment/labor can reflect crisis periods or large labor market shocks.
- Outliers should not be blindly deleted; many macro outliers are economically meaningful.

## 4. Time Series Analysis

We plot selected headline-style series, rolling means, rolling standard deviations, and stationarity tests.

In [7]:
def pick_representative_series(frame: pd.DataFrame, theme: str, max_per_bank: int = 1) -> pd.DataFrame:
    candidates = frame[frame["key_theme"] == theme].copy()
    ranked = (
        candidates.groupby(["central_bank_code", "indicator_key", "indicator"])
        .agg(observations=("value", "count"), importance=("importance", "min"))
        .reset_index()
        .sort_values(["central_bank_code", "importance", "observations"], ascending=[True, True, False])
    )
    picks = ranked.groupby("central_bank_code").head(max_per_bank)[["central_bank_code", "indicator_key"]]
    return candidates.merge(picks, on=["central_bank_code", "indicator_key"])

sample_ts = pd.concat([
    pick_representative_series(key_focus, "Inflation"),
    pick_representative_series(key_focus, "GDP/Growth"),
    pick_representative_series(key_focus, "Unemployment"),
])

fig = px.line(
    sample_ts.sort_values("date"),
    x="date",
    y="value_normalized",
    color="indicator",
    facet_col="central_bank_code",
    facet_col_wrap=2,
    title="Representative Normalized Time Series by Central Bank",
)
fig.update_layout(height=1100)
fig.show()

In [8]:
def rolling_view(frame: pd.DataFrame, central_bank: str, indicator_key: str, window: int = 6) -> pd.DataFrame:
    series = (
        frame[(frame["central_bank_code"] == central_bank) & (frame["indicator_key"] == indicator_key)]
        .sort_values("date")
        .copy()
    )
    series["rolling_mean"] = series["value_normalized"].rolling(window, min_periods=max(3, window // 2)).mean()
    series["rolling_std"] = series["value_normalized"].rolling(window, min_periods=max(3, window // 2)).std()
    return series

# Change these to inspect another central bank/indicator.
example_bank = "FED"
example_indicator = "cpi_headline_yoy" if "cpi_headline_yoy" in df[df.central_bank_code == "FED"].indicator_key.unique() else df[df.central_bank_code == "FED"].indicator_key.iloc[0]
rv = rolling_view(df, example_bank, example_indicator)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Normalized Value and Rolling Mean", "Rolling Standard Deviation"))
fig.add_trace(go.Scatter(x=rv["date"], y=rv["value_normalized"], name="Value"), row=1, col=1)
fig.add_trace(go.Scatter(x=rv["date"], y=rv["rolling_mean"], name="Rolling mean"), row=1, col=1)
fig.add_trace(go.Scatter(x=rv["date"], y=rv["rolling_std"], name="Rolling std"), row=2, col=1)
fig.update_layout(height=700, title=f"Rolling Statistics: {example_bank} / {example_indicator}")
fig.show()

In [9]:
def stationarity_result(values: pd.Series) -> dict:
    values = values.dropna().astype(float)
    result = {"observations": len(values), "adf_p_value": np.nan, "kpss_p_value": np.nan, "likely_stationary": None}
    if len(values) < 12 or values.nunique() < 4:
        return result
    try:
        result["adf_p_value"] = adfuller(values, autolag="AIC")[1]
    except Exception:
        pass
    try:
        result["kpss_p_value"] = kpss(values, regression="c", nlags="auto")[1]
    except Exception:
        pass
    if pd.notna(result["adf_p_value"]) and pd.notna(result["kpss_p_value"]):
        result["likely_stationary"] = bool(result["adf_p_value"] < 0.05 and result["kpss_p_value"] > 0.05)
    return result

stationarity_rows = []
for (bank, indicator_key), group in key_focus.sort_values("date").groupby(["central_bank_code", "indicator_key"]):
    row = {
        "central_bank_code": bank,
        "indicator_key": indicator_key,
        "indicator": group["indicator"].iloc[0],
        "key_theme": group["key_theme"].iloc[0],
    }
    row.update(stationarity_result(group["value"]))
    stationarity_rows.append(row)

stationarity_df = pd.DataFrame(stationarity_rows).sort_values(["key_theme", "central_bank_code", "indicator_key"])
stationarity_df.head(30)

,central_bank_code,indicator_key,indicator,key_theme,observations,adf_p_value,kpss_p_value,likely_stationary
4,BOC,gdp_mom,GDP (MoM),GDP/Growth,71,0.2067,0.1000,False
19,BOE,gdp_mom,GDP (MoM),GDP/Growth,69,0.4772,0.1000,False
20,BOE,gdp_qoq,GDP (QoQ),GDP/Growth,24,0.0001,0.0417,False
21,BOE,gdp_yoy,GDP (YoY),GDP/Growth,73,0.0000,0.1000,True
26,BOE,niesr_monthly_gdp_tracker,NIESR Monthly GDP Tracker,GDP/Growth,64,0.4006,0.1000,False
42,BOJ,gdp_qoq,GDP (QoQ),GDP/Growth,25,0.9746,0.1000,False
51,ECB,gdp_qoq,GDP (QoQ),GDP/Growth,24,0.1240,0.0417,False
112,SNB,gdp_qoq,GDP (QoQ),GDP/Growth,23,0.9965,0.0994,False
0,BOC,core_cpi_mom,Core CPI (MoM),Inflation,45,0.0000,0.0862,True
1,BOC,core_cpi_yoy,Core CPI (YoY),Inflation,76,0.4313,0.1000,False


### Stationarity interpretation

- ADF p-value below `0.05` suggests stationarity.
- KPSS p-value above `0.05` suggests stationarity.
- If both agree, the `likely_stationary` flag is more reliable.
- Non-stationary macro series often need differencing, year-over-year transforms, or regime-aware modeling.

## 5. Bivariate and Multivariate Relationships

We calculate correlations on normalized values. This avoids comparing raw levels with incompatible units.

In [10]:
def make_pivot(frame: pd.DataFrame, bank: str, min_points: int = 12) -> pd.DataFrame:
    pivot = (
        frame[frame["central_bank_code"] == bank]
        .pivot_table(index="date", columns="indicator_key", values="value_normalized", aggfunc="mean")
        .sort_index()
    )
    return pivot.dropna(axis=1, thresh=min_points)

bank = "FED"
pivot = make_pivot(key_focus, bank)
corr = pivot.corr(min_periods=18)

fig = px.imshow(
    corr,
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title=f"Correlation Matrix: {bank} Key Indicators",
)
fig.update_layout(height=900)
fig.show()

In [11]:
corr_named = corr.copy()
corr_named.index.name = "left_indicator"
corr_named.columns.name = "right_indicator"
corr_pairs = (
    corr_named.where(np.triu(np.ones(corr_named.shape), k=1).astype(bool))
    .stack()
    .rename("correlation")
    .reset_index()
    .rename(columns={"level_0": "left_indicator", "level_1": "right_indicator"})
)
corr_pairs["abs_correlation"] = corr_pairs["correlation"].abs()
corr_pairs.sort_values("abs_correlation", ascending=False).head(20)

,left_indicator,right_indicator,correlation,abs_correlation
322,cpi_headline_yoy,pce_price_index_yoy,0.9907,0.9907
267,core_ppi_yoy,ppi_yoy,0.9826,0.9826
436,export_prices_yoy,import_prices_yoy,0.9792,0.9792
95,core_cpi_yoy,core_pce_price_index_yoy,0.9714,0.9714
291,cpi_headline_mom,pce_price_index_mom,0.9627,0.9627
172,core_pce_price_index_yoy,pce_price_index_yoy,0.9542,0.9542
10,consumer_inflation_expectations,cpi_headline_yoy,0.9479,0.9479
22,consumer_inflation_expectations,pce_price_index_yoy,0.9429,0.9429
112,core_cpi_yoy,pce_price_index_yoy,0.9376,0.9376
262,core_ppi_yoy,pce_price_index_yoy,0.9360,0.9360


In [12]:
top_pair = corr_pairs.sort_values("abs_correlation", ascending=False).head(1)
if not top_pair.empty:
    left = top_pair.iloc[0]["left_indicator"]
    right = top_pair.iloc[0]["right_indicator"]
    scatter_df = pivot[[left, right]].dropna().reset_index()
    fig = px.scatter(
        scatter_df,
        x=left,
        y=right,
        trendline="ols",
        title=f"Scatter Plot: {bank} {left} vs {right}",
    )
    fig.show()

## 6. Lead-Lag Relationships

The function below calculates cross-correlations where the candidate indicator is shifted backward by lag periods. A positive lag means the candidate may lead the target.

In [13]:
def cross_correlation_by_lag(pivot: pd.DataFrame, target: str, candidate: str, max_lag: int = 6) -> pd.DataFrame:
    rows = []
    for lag in range(max_lag + 1):
        aligned = pd.concat([pivot[target], pivot[candidate].shift(lag)], axis=1).dropna()
        if len(aligned) >= 12:
            rows.append({"lag": lag, "correlation": aligned.iloc[:, 0].corr(aligned.iloc[:, 1]), "observations": len(aligned)})
    return pd.DataFrame(rows)

# Example: choose a headline target and a candidate with enough data.
available = pivot.columns.tolist()
target = next((x for x in available if "cpi" in x or "gdp" in x or "unemployment" in x), available[0])
candidate = next((x for x in available if x != target), available[1] if len(available) > 1 else available[0])
lag_df = cross_correlation_by_lag(pivot, target, candidate)

fig = px.bar(lag_df, x="lag", y="correlation", text="observations", title=f"Cross-Correlation by Lag: {candidate} -> {target}")
fig.show()
lag_df

,lag,correlation,observations
0,0,0.4950,34


## 7. Grouping and Aggregation

These summaries compare coverage, normalized means, volatility, and outlier rates by central bank, country, category, and frequency.

In [14]:
group_summary = (
    df.groupby(["central_bank_code", "country", "primary_category", "frequency"])
    .agg(
        observations=("value", "count"),
        indicators=("indicator_key", "nunique"),
        mean_normalized=("value_normalized", "mean"),
        volatility=("value_normalized", "std"),
        outliers=("is_outlier_iqr", "sum"),
        first_date=("date", "min"),
        last_date=("date", "max"),
    )
    .reset_index()
    .sort_values(["central_bank_code", "primary_category", "frequency"])
)
group_summary.head(40)

,central_bank_code,country,primary_category,frequency,observations,indicators,mean_normalized,volatility,outliers,first_date,last_date
0,BOC,CA,Growth,monthly,808,15,0.0000,0.9913,27,2019-11-01,2026-03-01
1,BOC,CA,Housing,monthly,128,2,0.0000,0.9961,0,2019-12-01,2026-03-01
2,BOC,CA,Inflation,monthly,569,8,0.0000,0.9938,0,2019-11-01,2026-03-01
3,BOC,CA,Labor,monthly,304,4,0.0000,0.9950,15,2019-12-01,2026-03-01
4,BOC,CA,Monetary Policy,monthly,2,1,0.0000,0.0000,0,2026-01-28,2026-03-18
5,BOC,CA,Sentiment,monthly,55,1,0.0000,1.0000,1,2021-10-01,2026-04-01
6,BOC,CA,Trade,monthly,228,3,0.0000,0.9956,0,2019-11-01,2026-02-01
7,BOE,UK,Growth,monthly,736,12,0.0000,0.9925,56,2019-11-01,2026-04-01
8,BOE,UK,Growth,quarterly,24,1,-0.0000,1.0000,4,2020-01-01,2025-10-01
9,BOE,UK,Housing,monthly,238,5,-0.0000,0.9915,0,2019-12-01,2026-04-22


In [15]:
fig = px.bar(
    group_summary,
    x="central_bank_code",
    y="observations",
    color="primary_category",
    facet_col="frequency",
    title="Observation Coverage by Central Bank, Category, and Frequency",
)
fig.show()

## 8. PCA

PCA is applied separately by central bank using normalized values. This helps identify whether a smaller number of latent factors explain much of the movement across indicators.

In [16]:
def run_pca_for_bank(frame: pd.DataFrame, bank: str, min_points: int = 12):
    pivot = make_pivot(frame, bank, min_points=min_points)
    filled = pivot.interpolate(limit_direction="both").ffill().bfill()
    filled = filled.dropna(axis=1)
    standardized = (filled - filled.mean()) / filled.std(ddof=0).replace(0, np.nan)
    standardized = standardized.dropna(axis=1)
    matrix = standardized.to_numpy(dtype=float)
    u, s, vh = np.linalg.svd(matrix, full_matrices=False)
    explained = s**2 / np.sum(s**2)
    loadings = pd.DataFrame(vh.T, index=standardized.columns, columns=[f"PC{i+1}" for i in range(vh.shape[0])])
    explained_df = pd.DataFrame({
        "component": [f"PC{i+1}" for i in range(len(explained))],
        "explained_variance_ratio": explained,
        "cumulative_explained_variance": np.cumsum(explained),
    })
    return explained_df, loadings

pca_bank = "FED"
explained_df, loadings = run_pca_for_bank(df, pca_bank)
explained_df.head(10)

,component,explained_variance_ratio,cumulative_explained_variance
0,PC1,0.3685,0.3685
1,PC2,0.1877,0.5562
2,PC3,0.1396,0.6958
3,PC4,0.0731,0.7689
4,PC5,0.0610,0.8299
5,PC6,0.0423,0.8722
6,PC7,0.0226,0.8948
7,PC8,0.0159,0.9108
8,PC9,0.0157,0.9265
9,PC10,0.0120,0.9384


In [17]:
fig = px.bar(
    explained_df.head(10),
    x="component",
    y="explained_variance_ratio",
    title=f"PCA Explained Variance: {pca_bank}",
)
fig.show()

top_loadings = (
    loadings[["PC1", "PC2", "PC3"]]
    .abs()
    .sort_values("PC1", ascending=False)
    .head(15)
)
top_loadings

,PC1,PC2,PC3
indicator_key,,,
pce_prices_qoq,0.1956,0.0445,0.0413
core_pce_prices_qoq,0.1869,0.0064,0.0735
import_prices_yoy,0.1792,0.0861,0.0287
ppi_yoy,0.1789,0.0295,0.1193
core_cpi_mom,0.1773,0.0145,0.0118
pce_price_index_mom,0.1765,0.0217,0.0714
cpi_headline_mom,0.1763,0.0238,0.0684
challenger_job_cuts,0.1754,0.0950,0.0898
core_ppi_yoy,0.1736,0.0010,0.1389


### PCA interpretation

- A high PC1 share means many indicators move together, suggesting a common macro factor.
- Loadings show which indicators drive each component.
- Highly similar loadings among related indicators may indicate redundancy or multicollinearity.

## 9. Structured Findings Template

Use the cells below after running the notebook to record the strongest findings for the modeling phase.

In [18]:
eda_findings = {
    "coverage": {
        "rows": len(df),
        "series": df[["central_bank_code", "indicator_key"]].drop_duplicates().shape[0],
        "date_start": str(df["date"].min().date()),
        "date_end": str(df["date"].max().date()),
    },
    "potential_modeling_issues": {
        "outlier_rows": int(df["is_outlier_iqr"].sum()),
        "high_missing_estimate_rows": int(df["estimate_value"].isna().sum()),
        "non_stationary_key_series": int((stationarity_df["likely_stationary"] == False).sum()),
    },
    "recommended_next_steps": [
        "Review high-correlation pairs before selecting model features.",
        "Prefer normalized values for cross-indicator comparisons.",
        "Use differencing or year-over-year transformations for non-stationary series.",
        "Treat macro outliers as event information first, not automatic data errors.",
        "Validate lead-lag findings out of sample before using them as predictive signals.",
    ],
}
eda_findings

{'coverage': {'rows': 15100,
  'series': 293,
  'date_start': '2019-10-01',
  'date_end': '2026-04-22'},
 'potential_modeling_issues': {'outlier_rows': 394,
  'high_missing_estimate_rows': 2155,
  'non_stationary_key_series': 74},
 'recommended_next_steps': ['Review high-correlation pairs before selecting model features.',
  'Prefer normalized values for cross-indicator comparisons.',
  'Use differencing or year-over-year transformations for non-stationary series.',
  'Treat macro outliers as event information first, not automatic data errors.',
  'Validate lead-lag findings out of sample before using them as predictive signals.']}